# Task 5 and 6

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn.datasets
from sklearn.preprocessing import StandardScaler

# Load and preprocess data (as given in assignment)
X, y = sklearn.datasets.load_breast_cancer(return_X_y=True)
X = StandardScaler().fit_transform(X)
y = 2 * y - 1  # {0,1} -> {-1,+1}

N, n = X.shape
X_tilde = np.hstack([X, np.ones((N, 1))])
print(f'N = {N}, n = {n}')

In [ ]:
# phi, sigma, and helpers

def phi(s):
    return np.logaddexp(0.0, s)

def sigma(s):
    # branch to avoid overflow in exp
    out = np.empty_like(s, dtype=float)
    pos = s >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-s[pos]))
    neg = ~pos
    ex = np.exp(s[neg])
    out[neg] = ex / (1.0 + ex)
    return out

def apply_D(theta):
    """D @ theta without building the full matrix. Just zeros the last entry (the bias)."""
    out = theta.copy()
    out[-1] = 0.0
    return out

def L(gamma):
    """Lipschitz constant from 4(b): gamma + ||X_tilde||^2 / 4"""
    return gamma + np.linalg.norm(X_tilde, ord=2)**2 / 4

def nll(theta):
    """Negative log-likelihood"""
    return np.sum(phi(-y * (X_tilde @ theta)))

def f0(theta, gamma):
    """Objective: -logL + gamma/2 * ||D theta||^2"""
    Dt = apply_D(theta)
    return nll(theta) + 0.5 * gamma * np.dot(Dt, Dt)

def grad_f0(theta, gamma):
    """Gradient from 4(a). Uses elementwise product instead of diag(y) matrix."""
    z = -y * (X_tilde @ theta)
    return -(X_tilde.T @ (y * sigma(z))) + gamma * apply_D(theta)

In [ ]:
# Gradient descent (Algorithm 6.4 in Hansson)

def gd(theta_0, gamma, max_iter=1000, tol=0.0):
    t = 1.0 / L(gamma)
    theta = theta_0.copy()
    obj_vals = [f0(theta, gamma)]

    for k in range(max_iter):
        g = grad_f0(theta, gamma)
        if np.linalg.norm(g) <= tol:
            break
        theta = theta - t * g
        obj_vals.append(f0(theta, gamma))

    return theta, np.array(obj_vals)

In [ ]:
# Accelerated gradient descent (Algorithm 6.8 in Hansson, smooth case h=0)

def agd(theta_0, gamma, max_iter=1000, tol=0.0, record=True):
    t = 1.0 / L(gamma)
    x = theta_0.copy()
    y_k = theta_0.copy()
    gam_k = 1.0  # momentum parameter (called gamma_k in the textbook)
    obj_vals = [f0(x, gamma)] if record else None

    for k in range(max_iter):
        x_new = y_k - t * grad_f0(y_k, gamma)
        gam_next = (1 + np.sqrt(1 + 4 * gam_k**2)) / 2
        y_k = x_new + ((gam_k - 1) / gam_next) * (x_new - x)
        x = x_new
        gam_k = gam_next

        if record:
            obj_vals.append(f0(x, gamma))
        if np.linalg.norm(grad_f0(x, gamma)) <= tol:
            break

    return x, (np.array(obj_vals) if record else None)

## Task 5

We apply both methods with $\gamma = 3$ and $\theta_0 = 0$ for 1000 iterations.

In [ ]:
gamma = 3.0
theta_0 = np.zeros(n + 1)

print(f'L = {L(gamma):.2f}, step size t = {1/L(gamma):.4e}')

theta_gd,  hist_gd  = gd(theta_0, gamma, max_iter=1000)
theta_agd, hist_agd = agd(theta_0, gamma, max_iter=1000)

print(f'Final objective GD:  {hist_gd[-1]:.6f}')
print(f'Final objective AGD: {hist_agd[-1]:.6f}')
print(f'GD is monotone:  {np.all(np.diff(hist_gd) <= 0)}')
print(f'AGD is monotone: {np.all(np.diff(hist_agd) <= 0)}')

In [ ]:
plt.figure(figsize=(6, 4))
plt.semilogy(hist_gd,  label='GD',  color='tab:blue')
plt.semilogy(hist_agd, label='AGD', color='tab:purple')
plt.xlabel('Iteration $k$')
plt.ylabel(r'$f_0(\theta_k)$')
plt.title('Objective value (log scale)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

GD is a descent method — the objective decreases monotonically, consistent with the bound $f(\theta_{k+1}) \le f(\theta_k) - \frac{1}{2L}\|\nabla f(\theta_k)\|^2$ from $L$-smoothness. The worst-case rate is $O(1/k)$ (cf.\ Book, Sec.~6.2.2).

AGD is *not* a descent method. The objective exhibits small oscillations characteristic of Algorithm 6.8, which evaluates the gradient at an extrapolated point without enforcing descent (cf.\ Book, Sec.~6.5.3). Its worst-case rate is $O(1/k^2)$. The difference in convergence rates is difficult to distinguish on this plot since both methods converge to similar values, but AGD reaches the final objective level significantly faster than GD.

## Task 6

We compute the trade-off curve for 20 values of $\gamma \in [10^{-3}, 10^3]$, warm-starting each solve from the previous solution as suggested in the hint.

In [ ]:
gammas = np.logspace(-3, 3, 20)

reg_terms = np.empty(len(gammas))
nll_terms = np.empty(len(gammas))

theta = np.zeros(n + 1)  # cold start only for first gamma
for i, g in enumerate(gammas):
    theta, _ = agd(theta, g, max_iter=50000, tol=1e-6, record=False)
    Dt = apply_D(theta)
    reg_terms[i] = 0.5 * np.dot(Dt, Dt)  # (1/2)||D theta*||^2
    nll_terms[i] = nll(theta)

print(f'Done. Gamma range: [{gammas[0]:.0e}, {gammas[-1]:.0e}]')

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(reg_terms, nll_terms, 'o-', color='tab:purple')
plt.xlabel(r'$\frac{1}{2}\|D\theta^\star\|_2^2$')
plt.ylabel(r'$-\ln\mathcal{L}(\theta^\star)$')
plt.title('Trade-off curve')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

The trade-off curve has an L shape. Small $\gamma$ means weak regularisation, so the model fits the data well (low NLL) but $\|w^\star\|$ is large. Large $\gamma$ shrinks $w^\star$ toward zero, reducing the regularisation term but increasing NLL since the model can no longer separate the classes. Each point on the curve is Pareto-optimal — you can't improve one objective without worsening the other.